## Ajuste de base de dados

In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime
import sys


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.preprocessing import SolarPreprocessor

from Modelos_regressivos.modelo.model import create_mimo_model
from Modelos_regressivos.src.dataset_module import SolarEfficientDataset
from Modelos_regressivos.loss.pi_loss import PhysicsGuidedLoss
from loss_function.mseloss import MaskedMSELoss

from src.analysis_k_factor import KFactorAnalyzer
from utils.quantile_mapping import quantile_mapping

In [2]:
CONFIG = {
    # --- 1. PATH DA BASE DE DADOS ---
    'csv_path': 'data/pv0.csv',
    
    # --- 2. DIVISÃO DA BASE DE TREINO, TESTE E VALIDAÇÃO ---
    'split_ratios': {'train': 0.8, 'val': 0.2}, 
    'test_year':2022,

    # --- 3. PRÉ-PROCESSAMENTO (Física & Mapeamento) ---
    'preprocessing': {
        'latitude': -23.56,
        'longitude': -46.73,
        'altitude': 0,
        'timezone': 'Etc/GMT+3',
        'nominal_power': 156.0,
        'start_year': 2015,
        'features_to_scale':['temp_amb','wind_speed'],
        #'pv_power_col_csv': 'Pot_BT', # <--- AVALIAR PARA RETIRAR
        
        # DOCUMENTAÇÃO VIVA: Mapeamento "De -> Para"
        # O Preprocessor usará isso para renomear as colunas internamente.
        # Chave (Esquerda): Nome como está no CSV bruto.
        # Valor (Direita): Nome padronizado usado no código.
        'column_mapping': {
            'Pot_BT': 'target',
            'Irradiação Global horária(horizontal) kWh/m2': 'ghi',
            'Irradiação Difusa horária kWh/m2': 'dhi',
            'Irradiação Global horária(Inclinada 27°) kWh/m2': 'irrad_poa',
            'Temperatura ambiente °C': 'temp_amb',
            'Umidade Relativa %': 'humidity',
            'Velocidade média do vento m/s': 'wind_speed'
        }
    },

    # --- 4. ESTRATÉGIA DE MODELAGEM ---
    # mode: 'sky' (prevê k) ou 'power' (prevê kW normalizado)
    'prediction_mode': 'sky',
    
    # Qual variável o modelo vai prever? ('k' ou 'target')
    'target_col': ['kt', 'fracao_difusa'], 
    
    # Features de entrada
    'feature_cols': [
        'sin_azimuth', 'elevation', 'temp_amb', 'humidity'
    ],
    
    'aux_col':[
        'ghi_cs', 'cos_zenith', 
        'elevation', 'ghi_extra'
    ],

    # --- 5. ARQUITETURA E TREINO ---
    'model_type': 'Teste_2',
    'cell_type': 'lstm',
    'input_seq_len': 24,
    'output_seq_len': 1,
    'hidden_sizes': [128, 64],
    'learning_rate': 0.001,
    'batch_size': 32,
    'epochs': 50,
    'dropout': 0.1,
    'bidirectional': False,
    'use_attention': False,
    'use_feature_attention': False,
    'patience': 10,
    'loss_function':'mse',      # "cpi_loss", "mse"
    'use_mask':True
}

OUTPUT_ROOT = 'trained_models'
ARTIFACTS_DIR = 'artifacts'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 1. Setup
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
exp_name = f"{timestamp}_{CONFIG['model_type']}"
exp_dir = os.path.join(OUTPUT_ROOT, exp_name)
os.makedirs(exp_dir, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 2. Leitura
#csv_path = CONFIG['data/pv0.csv']
#print(f"⏳ Lendo: {csv_path}")
df = pd.read_csv('/workspaces/Remodelacao_mestrado/data/pv0.csv')


if 'Date_Time' in df.columns:
    df['Date_Time'] = pd.to_datetime(df['Date_Time'])
    #df['Date_Time'] -=  pd.Timedelta(minutes=30)
    df = df.drop_duplicates(subset=['Date_Time'], keep='first').set_index('Date_Time').sort_index()
df = df[~df.index.duplicated(keep='first')]

# 3. Pré-processamento
pp_conf = CONFIG['preprocessing']

# Instancia passando o mapa explícito. 
# Isso garante que a padronização aconteça conforme o CONFIG acima.
preprocessor = SolarPreprocessor(
    latitude=pp_conf['latitude'], 
    longitude=pp_conf['longitude'], 
    altitude=pp_conf['altitude'],
    timezone=pp_conf['timezone'], 
    nominal_power=pp_conf['nominal_power'], 
    start_year=pp_conf['start_year'],
    cs_model = 'esra',
    features_to_scale=pp_conf['features_to_scale'],
    target_col=CONFIG['prediction_mode'], # <--- unica variavel que não vem do preprocessing
    column_mapping=pp_conf['column_mapping'],
    kasten_corr=True
)

preprocessor.fit(df)
preprocessor.save_scalers(exp_dir)
preprocessor.save_scalers(ARTIFACTS_DIR)

# O método transform usa o column_mapping para renomear as colunas
df_processed = preprocessor.transform(df)

💾 Scalers salvos em: trained_models/2026-03-10_19-55-45_Teste_2
💾 Scalers salvos em: artifacts
Otimizando TL para 381 dias selecionados...
Otimização concluída. Média TL: 6.28


In [4]:
last_year = CONFIG['test_year']

test_df = df_processed[df_processed.index.year == last_year].copy()
dev_df = df_processed[df_processed.index.year < last_year].copy()

n_dev = len(dev_df)
train_end = int(n_dev * CONFIG['split_ratios']['train'])

train_df = dev_df.iloc[:train_end].copy()
val_df = dev_df.iloc[train_end:].copy()

train_dataset = SolarEfficientDataset(
    df=train_df, 
    u_cols = CONFIG['feature_cols'],
    y_past_cols = CONFIG['target_col'],
    target_cols = CONFIG['target_col'],
    aux_cols = CONFIG['aux_col'],
    n_past=CONFIG['input_seq_len'], 
    n_future=CONFIG['output_seq_len']
)
val_dataset = SolarEfficientDataset(
    df=val_df, 
    u_cols = CONFIG['feature_cols'],
    y_past_cols = CONFIG['target_col'],
    target_cols = CONFIG['target_col'],
    aux_cols = CONFIG['aux_col'],
    n_past=CONFIG['input_seq_len'], 
    n_future=CONFIG['output_seq_len']
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

✅ Dataset pronto. Amostras válidas: 15646
✅ Dataset pronto. Amostras válidas: 4561


## Treinamento

In [7]:
# Criando o modelo
model = create_mimo_model(
    model_type='ARX', 
    n_u=len(CONFIG['feature_cols']), 
    n_y=len(CONFIG['target_col']), 
    n_future=CONFIG['output_seq_len'],
    order=CONFIG['input_seq_len'], 
    hidden_dim=128
)

# Movendo para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#criterion = PhysicsGuidedLoss(lambda_hard=0.1, lambda_soft=0.1, data_loss_type='mse').to(device)
criterion = MaskedMSELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

# --- LOOP DE TREINO E VALIDAÇÃO ---
for epoch in range(CONFIG['epochs']):
    # TREINAMENTO
    model.train()
    train_loss = 0.0
    
    for u_hist, y_hist, y_future, mask, aux_future in train_loader:
        # Move os dados para GPU/CPU
        u_hist, y_hist = u_hist.to(device), y_hist.to(device)
        y_future, mask, aux_future = y_future.to(device), mask.to(device), aux_future.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass direto com as variáveis separadas
        y_pred = model(u_hist, y_hist)
        
        # Cálculo da Loss
        loss, _ = criterion(y_pred, y_future)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # VALIDAÇÃO
    model.eval()
    val_loss = 0.0
    val_mse = 0.0
    
    with torch.no_grad():
        for u_hist, y_hist, y_future, mask, aux_future in val_loader:
            u_hist, y_hist = u_hist.to(device), y_hist.to(device)
            y_future, mask, aux_future = y_future.to(device), mask.to(device), aux_future.to(device)
            
            y_pred = model(u_hist, y_hist)
            loss, loss_dict = criterion(y_pred, y_future, aux_future, mask)
            
            val_loss += loss.item()
            val_mse += loss_dict['mse_stat']
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_mse = val_mse / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] | Train Loss: {avg_train_loss:.4f} | Val Loss (PINN): {avg_val_loss:.4f} | Val MSE: {avg_val_mse:.4f}")

print("Treinamento finalizado.")

TypeError: iteration over a 0-d tensor

## Teste

In [ ]:
test_dataset = SolarEfficientDataset(
    df=test_df, 
    u_cols = CONFIG['feature_cols'],
    y_past_cols = CONFIG['target_col'],
    target_cols = CONFIG['target_col'],
    aux_cols = CONFIG['aux_col'],
    n_past=CONFIG['input_seq_len'], 
    n_future=CONFIG['output_seq_len']
)

test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

✅ Dataset pronto. Amostras válidas: 3155


In [ ]:
def evaluate_test_set(model, test_loader, test_dataset, device):
    """
    Avalia o modelo no conjunto de teste e retorna um DataFrame indexado pelo Datetime.
    
    IMPORTANTE: O test_loader DEVE ter sido instanciado com shuffle=False!
    """
    model.eval()
    
    all_predictions = []
    all_targets = []
    
    # 1. Loop de Inferência (apenas tensores)
    with torch.no_grad():
        for u_hist, y_hist, y_future, mask, aux_future in test_loader:
            
            u_hist = u_hist.to(device)
            y_hist = y_hist.to(device)
            
            y_pred = model(u_hist, y_hist)
            
            all_predictions.append(y_pred.squeeze(1).cpu().numpy())
            all_targets.append(y_future.squeeze(1).cpu().numpy())
            
    # 2. Concatena os resultados
    preds_np = np.concatenate(all_predictions, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)
    
    # 3. Extrai os Timestamps exatos do Dataset
    # Como o DataLoader não embaralhou os dados, a ordem das previsões é 
    # exatamente a mesma ordem dos valid_indices do dataset.
    valid_idx = test_dataset.valid_indices
    timestamps = test_dataset.timestamps[valid_idx]
    
    # 4. Monta o DataFrame final
    df_results = pd.DataFrame({
        'kt_pred_pinn': preds_np[:, 0], # Sufixo para não confundir no merge
        'kd_pred_pinn': preds_np[:, 1],
        'kt_real': targets_np[:, 0],
        'kd_real': targets_np[:, 1]
    }, index=timestamps)
    
    # Dá um nome ao índice para garantir que o merge/join funcione perfeitamente
    df_results.index.name = 'datetime'
    
    return df_results

# ==========================================
# Exemplo de Uso e Merge
# ==========================================

# Certifique-se de passar o dataset e o loader
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df_pinn = evaluate_test_set(model, test_loader, test_dataset, device)

print(df_pinn.head())

# Supondo que você tenha outro dataframe df_skforecast com as previsões da base de teste
# df_skforecast.index também deve ser o datetime
# df_comparacao = df_pinn.join(df_skforecast[['kt_pred_rf', 'kd_pred_rf']], how='inner')

                           kt_pred_pinn  kd_pred_pinn   kt_real   kd_real
datetime                                                                 
2022-01-02 07:00:00-03:00     -0.109544      1.211979  0.124534  0.935801
2022-01-02 08:00:00-03:00     -0.130631      1.274023  0.228168  0.948632
2022-01-02 09:00:00-03:00     -0.080981      1.269686  0.504425  0.694458
2022-01-02 10:00:00-03:00      0.109429      1.115092  0.460698  0.784860
2022-01-02 11:00:00-03:00      0.072182      1.156405  0.481169  0.472487


## Analise de Resultados

In [ ]:
path = "/workspaces/Remodelacao_mestrado/analysis_outputs/Fisica_Atmosferica/2026-01-28_17-05-32_Teste_kt_kd/tabela.csv"
df_results = pd.read_csv(path, index_col = 0)

df_results.index = pd.to_datetime(df_results.index)

df_results['kt_pred_corr'] = quantile_mapping(df_results['kt_real'], df_results['kt_pred'])
df_results['kd_pred_corr'] = quantile_mapping(df_results['kd_real'], df_results['kd_pred'])

In [ ]:
df_results = pd.merge(df_results, df_pinn[['kt_pred_pinn', 'kd_pred_pinn']], 
                      left_index=True, right_index=True, how='inner')

df_results['kt_pred_pinn_corr'] = quantile_mapping(df_results['kt_real'], df_results['kt_pred_pinn'])
df_results['kd_pred_pinn_corr'] = quantile_mapping(df_results['kd_real'], df_results['kd_pred_pinn'])

In [ ]:
# 2. INSTÂNCIA DO NOVO ANALYZER
# Passamos o nome do modelo para criar a pasta organizada
k_analyzer = KFactorAnalyzer(model_name='analise_modelos_sky', output_dir="analysis_outputs")

# 3. CHAMADA DA FIGURA DE 3 PLOTS
# Esta função vai gerar o arquivo .png com os dados de teste
k_analyzer.plot_kt_kd_relationship(df_results)
k_analyzer.plot_clear_sky_day_analysis(df_results)
k_analyzer.plot_high_variability_day_analysis(df_results)
k_analyzer.plot_overcast_day_analysis(df_results)
#k_analyzer.calculate_statistical_metrics(df_results)
k_analyzer.plot_scatter_validation(df_results)
k_analyzer.plot_transient_day_analysis(df_results)

   ☀️ Dia selecionado (10 amostras): 2022-11-17
   ⛈️ Dia selecionado para alta variabilidade: 2022-10-19
   ☁️ Dia selecionado como Encoberto: 2022-11-02
   📈 Gerando matriz de scatter de validação estatística...
   🌦️ Dia Transiente selecionado: 2022-01-03
